# Liu2024 Locked Shallow + Same-Fold Short-Scale Riemann Fusion

Reuses the locked ShallowFBCSPNet outer-OOF probabilities and recomputes motor13 1 s/2 s Riemannian probabilities on the exact same outer folds. No neural model is trained or selected.

# 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime
import builtins
import hashlib
import json
import os
import platform
import random
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import signal
from scipy.special import expit
from scipy.stats import wilcoxon
from sklearn.covariance import OAS
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from pyriemann.tangentspace import TangentSpace

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Working directory: {Path.cwd()}")

# 2. Configuration
## 2.1 Locked Domain Defaults

The ordered Liu29 montage drops recorded CPz (raw index 17). Motor13 is the fixed symmetric subset used by the confirmed short-scale protocol.

In [ ]:
LIU29 = ["Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4", "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2"]
LIU29_RAW_INDICES = list(range(17)) + list(range(18, 30))
MOTOR13 = ["F3", "F4", "FCz", "FC3", "FC4", "Cz", "C3", "C4", "CP3", "CP4", "Pz", "P3", "P4"]
MOTOR13_RAW_INDICES = [LIU29_RAW_INDICES[LIU29.index(name)] for name in MOTOR13]
EXPECTED_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 51)]
assert len(LIU29) == len(LIU29_RAW_INDICES) == 29
assert len(MOTOR13) == len(MOTOR13_RAW_INDICES) == 13

## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-shallow-riemann-same-fold-fusion"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "locked_shallow_artifact": str(WORKING_DIR / "artifacts" / "liu2024-compact-mi-models" / "20260712_165746_790145_fd8ab986"),
    "experiment_name": "locked_shallow_motor13_short_scale_same_fold_fusion",
    "config_note": "Prespecified 50/50 probability fusion using locked Shallow OOF probabilities and same-fold motor13 short-scale Riemann predictions.",

    # ------------------------------------------------------------------
    # Dataset / immutable provenance
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "expected_subject_count": 50,
    "expected_trials_per_subject": 40,
    "expected_shallow_run_id": "20260712_165746_790145_fd8ab986",
    "expected_shallow_experiment_name": "compact_mi_locked_shallowfbcspnet",
    "expected_shallow_model_name": "ShallowFBCSPNet",
    "expected_channel_set": "liu29",
    "expected_liu29_names": LIU29,
    "expected_global_split_hash": "801ec1d2c981335f",
    "marker_channel_index": 32,
    "onset_marker_value": 2,
    "onset_plausible_range": [800, 1300],
    "onset_fallback_sample": 1003,
    "native_sfreq": 500,
    "mi_window_s": [0.0, 4.0],

    # ------------------------------------------------------------------
    # Fixed Riemann preprocessing / model
    # ------------------------------------------------------------------
    "spatial_mode": "motor13",
    "filter_context": "full_trial_then_crop",
    "average_reference": True,
    "bands_hz": [[8.0, 12.0], [13.0, 20.0], [20.0, 30.0], [8.0, 30.0]],
    "temporal_scales": {"1s": {"length_s": 1.0, "starts_s": [0.0, 1.0, 2.0, 3.0]}, "2s": {"length_s": 2.0, "starts_s": [0.0, 1.0, 2.0]}},
    "filter_order": 4,
    "covariance_estimator": "oas",
    "trace_normalize": True,
    "tangent_metric": "riemann",
    "base_classifier": "tangent_lda",
    "lda_solver": "lsqr",
    "lda_shrinkage": "auto",
    "riemann_probability_mapping": "sigmoid_of_training_margin_scaled_score",
    "margin_scale_eps": 1e-8,

    # ------------------------------------------------------------------
    # Evaluation / fixed fusion
    # ------------------------------------------------------------------
    "evaluation_mode": "reuse_locked_stratified_5fold",
    "cv_folds": 5,
    "cv_random_state": 2026,
    "shallow_weight": 0.5,
    "riemann_weight": 0.5,
    "bootstrap_iterations": 10000,
    "bootstrap_seed": 202607,

    # ------------------------------------------------------------------
    # Reproducibility / diagnostics
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "collapse_threshold": 0.9,
    "fail_on_any_fold_error": True,
    "run_synthetic_checks": False,
}


In [ ]:
if CONFIG["subjects_to_use"] is not None:
    raise ValueError("Primary same-fold fusion is locked to all 50 subjects")
if not np.isclose(CONFIG["shallow_weight"] + CONFIG["riemann_weight"], 1.0):
    raise ValueError("Fusion weights must sum to one")
if CONFIG["spatial_mode"] != "motor13" or CONFIG["filter_context"] != "full_trial_then_crop":
    raise ValueError("This notebook is fail-closed to the confirmed motor13 full-trial-then-crop protocol")
if CONFIG["covariance_estimator"] != "oas" or CONFIG["base_classifier"] != "tangent_lda":
    raise ValueError("This notebook is fail-closed to OAS covariance and tangent shrinkage-LDA")
print(json.dumps({k: CONFIG[k] for k in ["experiment_name", "locked_shallow_artifact", "expected_global_split_hash", "spatial_mode", "filter_context", "bands_hz", "temporal_scales"]}, indent=2))

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass
    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")

# 3. Load and Prepare Data
## 3.1 Locked Shallow Provenance and Split Assertions

Validation occurs before raw covariance work. The notebook refuses missing, duplicated, reordered, non-finite, non-normalized, or split-mismatched Shallow predictions.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def stable_hash(payload, length=16):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:length]

def load_json(path):
    with open(path, encoding="utf-8") as stream:
        return json.load(stream)

def validate_locked_shallow_artifact():
    root = Path(CONFIG["locked_shallow_artifact"]).resolve()
    required = ["config.json", "run_metadata.json", "subject_inventory.csv", "splits.json", "cv_results.json"]
    missing = [name for name in required if not (root / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Locked Shallow artifact is incomplete: {missing}")
    shallow_cfg = load_json(root / "config.json")
    metadata = load_json(root / "run_metadata.json")
    inventory = pd.read_csv(root / "subject_inventory.csv")
    splits = load_json(root / "splits.json")
    folds = load_json(root / "cv_results.json")

    assert metadata.get("run_id") == CONFIG["expected_shallow_run_id"]
    assert metadata.get("experiment_name") == CONFIG["expected_shallow_experiment_name"]
    assert metadata.get("model_name") == CONFIG["expected_shallow_model_name"]
    assert shallow_cfg.get("model_name") == CONFIG["expected_shallow_model_name"]
    assert shallow_cfg.get("channel_set") == CONFIG["expected_channel_set"]
    assert metadata.get("channel_names") == CONFIG["expected_liu29_names"]
    assert metadata.get("subjects") == EXPECTED_SUBJECTS
    assert len(inventory) == CONFIG["expected_subject_count"]
    assert inventory["subject_id"].tolist() == EXPECTED_SUBJECTS
    assert inventory["n_trials"].eq(CONFIG["expected_trials_per_subject"]).all()
    assert set(inventory["split_hash"].astype(str)) == {CONFIG["expected_global_split_hash"]}
    assert list(splits) == EXPECTED_SUBJECTS
    assert len(folds) == CONFIG["expected_subject_count"] * CONFIG["cv_folds"]

    fold_lookup = {}
    assertions = []
    for sid in EXPECTED_SUBJECTS:
        subject_splits = splits[sid]
        assert len(subject_splits) == CONFIG["cv_folds"]
        assert stable_hash(subject_splits) == CONFIG["expected_global_split_hash"]
        seen = []
        for split in subject_splits:
            fold_id = int(split["fold_id"])
            train_idx = np.asarray(split["train_indices"], dtype=int)
            test_idx = np.asarray(split["test_indices"], dtype=int)
            assert len(train_idx) == 32 and len(test_idx) == 8
            assert len(np.intersect1d(train_idx, test_idx)) == 0
            assert sorted(np.r_[train_idx, test_idx].tolist()) == list(range(40))
            seen.extend(test_idx.tolist())
            fold_lookup[(sid, fold_id)] = {"train_indices": train_idx, "test_indices": test_idx}
            assertions.append({"subject_id": sid, "fold_id": fold_id, "n_train": 32, "n_test": 8, "no_overlap": True, "complete_partition": True})
        assert sorted(seen) == list(range(40)) and len(set(seen)) == 40

    shallow_by_fold = {}
    for row in folds:
        key = (str(row["subject_id"]), int(row["fold_id"]))
        assert key in fold_lookup and key not in shallow_by_fold
        expected_test = fold_lookup[key]["test_indices"]
        test_idx = np.asarray(row["test_indices"], dtype=int)
        labels = np.asarray(row["true_labels"], dtype=int)
        probabilities = np.asarray(row["probabilities"], dtype=float)
        predictions = np.asarray(row["predictions"], dtype=int)
        assert np.array_equal(test_idx, expected_test)
        assert labels.shape == predictions.shape == (8,)
        assert probabilities.shape == (8, 2)
        assert np.all(np.isfinite(probabilities)) and np.all(probabilities >= 0.0) and np.all(probabilities <= 1.0)
        assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
        assert np.array_equal(predictions, probabilities.argmax(axis=1))
        shallow_by_fold[key] = {"test_indices": test_idx, "labels": labels, "probabilities": probabilities}
    assert set(shallow_by_fold) == set(fold_lookup)

    provenance = {
        "locked_artifact": str(root),
        "required_file_sha256": {name: sha256_file(root / name) for name in required},
        "run_id": metadata["run_id"],
        "experiment_name": metadata["experiment_name"],
        "model_name": metadata["model_name"],
        "channel_set": shallow_cfg["channel_set"],
        "channel_names": metadata["channel_names"],
        "global_split_hash": CONFIG["expected_global_split_hash"],
        "n_subjects": len(splits),
        "n_folds": len(folds),
        "n_predictions": int(sum(len(v["test_indices"]) for v in shallow_by_fold.values())),
        "shallow_retrained": False,
        "shallow_selected": False,
    }
    return fold_lookup, shallow_by_fold, assertions, provenance

LOCKED_SPLITS, SHALLOW_BY_FOLD, SPLIT_ASSERTIONS, SHALLOW_PROVENANCE = validate_locked_shallow_artifact()
print(f"Validated locked Shallow artifact: {len(SHALLOW_BY_FOLD)} folds, {SHALLOW_PROVENANCE['n_predictions']} OOF predictions")

## 3.2 Raw Trial Loading and Preprocessing

Each complete 8 s motor13 trial is average-referenced and filtered independently. The marker-relative 0-4 s MI segment is cropped only after filtering; no filter crosses a trial boundary.

In [ ]:
def selected_files():
    files = sorted(Path(CONFIG["source_extract_dir"]).glob("sub-*/sub-*_task-motor-imagery_eeg.mat"))
    if [p.parent.name for p in files] != EXPECTED_SUBJECTS:
        raise AssertionError(f"Expected ordered full50 raw files, got {[p.parent.name for p in files]}")
    return files

def load_subject_raw(path):
    sid = path.parent.name
    eeg = sio.loadmat(path)["eeg"][0, 0]
    raw = np.asarray(eeg["rawdata"], dtype=np.float64)
    labels = np.asarray(eeg["label"]).reshape(-1).astype(int)
    if set(np.unique(labels)).issubset({1, 2}):
        labels = labels - 1
    if raw.shape[0] != 40 or raw.shape[1] <= CONFIG["marker_channel_index"] or labels.shape != (40,):
        raise ValueError(f"Unexpected raw shape for {sid}: raw={raw.shape}, labels={labels.shape}")
    if np.bincount(labels, minlength=2).tolist() != [20, 20]:
        raise ValueError(f"Expected balanced binary labels for {sid}")
    marker = raw[:, CONFIG["marker_channel_index"], :]
    lo, hi = CONFIG["onset_plausible_range"]
    detected = []
    for trial_marker in marker:
        hits = np.flatnonzero(trial_marker == CONFIG["onset_marker_value"])
        valid = hits[(hits >= lo) & (hits <= hi)]
        detected.append(int(valid[0]) if len(valid) else -1)
    plausible = [value for value in detected if lo <= value <= hi]
    fallback = int(np.median(plausible)) if plausible else int(CONFIG["onset_fallback_sample"])
    onsets = np.asarray([value if lo <= value <= hi else fallback for value in detected], dtype=int)
    n_mi = int(round((CONFIG["mi_window_s"][1] - CONFIG["mi_window_s"][0]) * CONFIG["native_sfreq"]))
    if any(onset < 0 or onset + n_mi > raw.shape[-1] for onset in onsets):
        raise ValueError(f"MI crop outside trial for {sid}")
    trials = raw[:, MOTOR13_RAW_INDICES, :].copy()
    return {"subject_id": sid, "trials": trials, "labels": labels, "onsets": onsets, "fallback_count": int(np.sum(np.asarray(detected) < 0)), "source_path": str(path), "source_sha256": sha256_file(path)}

def build_view_specs():
    specs = []
    for scale, temporal in CONFIG["temporal_scales"].items():
        for start in temporal["starts_s"]:
            for band in CONFIG["bands_hz"]:
                specs.append({"id": f"{scale}_{start:g}s_{band[0]:g}-{band[1]:g}Hz", "scale": scale, "start_s": float(start), "length_s": float(temporal["length_s"]), "band_hz": [float(band[0]), float(band[1])]})
    return specs

VIEW_SPECS = build_view_specs()
assert sum(view["scale"] == "1s" for view in VIEW_SPECS) == 16
assert sum(view["scale"] == "2s" for view in VIEW_SPECS) == 12

def estimate_oas_covariance(window):
    fitted = OAS(store_precision=False, assume_centered=True).fit(window.T)
    covariance = fitted.covariance_
    if CONFIG["trace_normalize"]:
        trace = float(np.trace(covariance))
        if not np.isfinite(trace) or trace <= 0:
            raise ValueError("Invalid covariance trace")
        covariance = covariance / trace
    eigenvalues = np.maximum(np.linalg.eigvalsh(covariance), np.finfo(float).eps)
    probabilities = eigenvalues / eigenvalues.sum()
    diagnostics = {"shrinkage": float(fitted.shrinkage_), "effective_rank": float(np.exp(-np.sum(probabilities * np.log(probabilities)))), "condition": float(eigenvalues[-1] / eigenvalues[0])}
    return covariance, diagnostics

def extract_covariance_views(trials, onsets):
    if trials.shape[1] != 13:
        raise AssertionError("Motor13 must contain exactly 13 channels")
    covariances = np.empty((len(trials), len(VIEW_SPECS), 13, 13), dtype=np.float64)
    diagnostic_rows = []
    sos_by_band = {tuple(band): signal.butter(CONFIG["filter_order"], band, btype="bandpass", fs=CONFIG["native_sfreq"], output="sos") for band in CONFIG["bands_hz"]}
    for trial_index, trial in enumerate(trials):
        referenced = trial - trial.mean(axis=0, keepdims=True)
        filtered = {band: signal.sosfiltfilt(sos, referenced, axis=-1) for band, sos in sos_by_band.items()}
        for view_index, view in enumerate(VIEW_SPECS):
            mi_start = int(onsets[trial_index] + round(CONFIG["mi_window_s"][0] * CONFIG["native_sfreq"]))
            start = mi_start + int(round(view["start_s"] * CONFIG["native_sfreq"]))
            stop = start + int(round(view["length_s"] * CONFIG["native_sfreq"]))
            window = filtered[tuple(view["band_hz"])][:, start:stop]
            expected_samples = int(round(view["length_s"] * CONFIG["native_sfreq"]))
            if window.shape != (13, expected_samples):
                raise ValueError(f"Incomplete {view['id']} window for trial {trial_index}")
            covariance, diagnostics = estimate_oas_covariance(window)
            if not np.all(np.isfinite(covariance)):
                raise ValueError("Non-finite covariance")
            covariances[trial_index, view_index] = covariance
            diagnostic_rows.append({"trial_index": trial_index, "view_id": view["id"], **diagnostics})
    return covariances, diagnostic_rows

# 4. Model
## 4.1 Fold-Local Tangent Shrinkage-LDA and Fixed Probability Fusion

In [ ]:
def oriented_decision_score(model, features):
    score = np.asarray(model.decision_function(features), dtype=float).reshape(-1)
    classes = np.asarray(model.classes_)
    if classes.tolist() == [0, 1]:
        return score
    if classes.tolist() == [1, 0]:
        return -score
    raise ValueError(f"Unexpected classifier classes: {classes.tolist()}")

def robust_margin_scale(training_scores):
    training_scores = np.asarray(training_scores, dtype=float)
    scale = float(np.median(np.abs(training_scores)))
    if not np.isfinite(scale) or scale < CONFIG["margin_scale_eps"]:
        scale = float(np.sqrt(np.mean(training_scores ** 2)))
    if not np.isfinite(scale) or scale < CONFIG["margin_scale_eps"]:
        raise ValueError("Degenerate outer-training margins")
    return scale

def fit_view_probability(covariances, labels, train_indices, test_indices):
    train_covariances = covariances[train_indices]
    test_covariances = covariances[test_indices]
    train_labels = labels[train_indices]
    tangent = TangentSpace(metric=CONFIG["tangent_metric"]).fit(train_covariances)
    train_features = tangent.transform(train_covariances)
    test_features = tangent.transform(test_covariances)
    scaler = StandardScaler().fit(train_features)
    train_features = scaler.transform(train_features)
    test_features = scaler.transform(test_features)
    classifier = LinearDiscriminantAnalysis(solver=CONFIG["lda_solver"], shrinkage=CONFIG["lda_shrinkage"]).fit(train_features, train_labels)
    training_scores = oriented_decision_score(classifier, train_features)
    margin_scale = robust_margin_scale(training_scores)
    test_scores = oriented_decision_score(classifier, test_features) / margin_scale
    probability_1 = expit(test_scores)
    probabilities = np.column_stack([1.0 - probability_1, probability_1])
    return probabilities, test_scores, margin_scale

def scores_to_probabilities(scores):
    probability_1 = expit(np.asarray(scores, dtype=float))
    return np.column_stack([1.0 - probability_1, probability_1])

def fit_short_scale_riemann(covariances, labels, train_indices, test_indices):
    by_scale = {}
    margin_scales = {}
    for scale in ["1s", "2s"]:
        view_indices = [i for i, view in enumerate(VIEW_SPECS) if view["scale"] == scale]
        view_scores = []
        for view_index in view_indices:
            probabilities, scores, fitted_scale = fit_view_probability(covariances[:, view_index], labels, train_indices, test_indices)
            view_scores.append(scores)
            margin_scales[VIEW_SPECS[view_index]["id"]] = fitted_scale
        scale_scores = np.mean(np.stack(view_scores), axis=0)
        by_scale[scale] = {"probabilities": scores_to_probabilities(scale_scores), "scores": scale_scores}
    equal_scores = 0.5 * (by_scale["1s"]["scores"] + by_scale["2s"]["scores"])
    equal_probabilities = scores_to_probabilities(equal_scores)
    return by_scale, equal_probabilities, equal_scores, margin_scales

def fixed_fusion(shallow_probabilities, riemann_probabilities):
    fused = CONFIG["shallow_weight"] * shallow_probabilities + CONFIG["riemann_weight"] * riemann_probabilities
    if not np.allclose(fused.sum(axis=1), 1.0, atol=1e-10):
        raise AssertionError("Fused probabilities are not normalized")
    return fused

## 4.2 Synthetic Unit Checks

In [ ]:
def run_synthetic_checks():
    rng = np.random.default_rng(202607)
    labels = np.repeat([0, 1], 16)
    covariances = []
    for label in labels:
        samples = rng.normal(size=(200, 5))
        samples[:, 0] *= 1.0 + 0.35 * label
        fitted = OAS(store_precision=False, assume_centered=True).fit(samples)
        covariance = fitted.covariance_ / np.trace(fitted.covariance_)
        covariances.append(covariance)
    covariances = np.asarray(covariances)
    train_indices, test_indices = np.arange(24), np.arange(24, 32)
    probabilities, scores, scale = fit_view_probability(covariances, labels, train_indices, test_indices)
    assert probabilities.shape == (8, 2)
    assert scores.shape == (8,)
    assert np.all(np.isfinite(probabilities)) and np.allclose(probabilities.sum(1), 1.0)
    assert scale > 0
    shallow = np.full((8, 2), 0.5)
    fused = fixed_fusion(shallow, probabilities)
    assert np.allclose(fused, 0.5 * shallow + 0.5 * probabilities)
    return {"probability_shape": list(probabilities.shape), "finite": True, "normalized": True, "positive_training_margin_scale": True, "fixed_fusion": True}

if CONFIG["run_synthetic_checks"]:
    print(run_synthetic_checks())

# 5. Training
## 5.1 Same-Fold Evaluation Runner

There is no neural training. Every Riemann tangent reference, scaler, margin scale, and classifier is fit on the locked outer-training indices only. Trial-local OAS covariance uses only samples from its own trial.

In [ ]:
METHODS = ["shallow_locked", "riemann_1s", "riemann_2s", "riemann_equal_short_scales", "shallow_riemann_fusion"]

def fold_metric_record(subject_id, fold_id, method, train_indices, test_indices, labels, probabilities):
    predictions = probabilities.argmax(axis=1)
    majority = float(max(np.mean(predictions == 0), np.mean(predictions == 1)))
    return {"subject_id": subject_id, "fold_id": int(fold_id), "method": method, "train_indices": train_indices.tolist(), "test_indices": test_indices.tolist(), "true_labels": labels.tolist(), "predictions": predictions.tolist(), "probabilities": probabilities.tolist(), "accuracy": float(accuracy_score(labels, predictions)), "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)), "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(), "prediction_histogram": np.bincount(predictions, minlength=2).tolist(), "collapse_diagnostics": {"majority_prediction_fraction": majority, "collapsed": bool(majority >= CONFIG["collapse_threshold"])}, "outer_test_used_for_selection": False, "outer_test_used_for_fit": False, "shallow_retrained": False}

def run_subject(subject_data):
    sid = subject_data["subject_id"]
    labels = subject_data["labels"]
    covariances, covariance_diagnostics = extract_covariance_views(subject_data["trials"], subject_data["onsets"])
    fold_results, prediction_rows, margin_rows = [], [], []
    seen = {method: np.zeros(len(labels), dtype=int) for method in METHODS}
    for fold_id in range(CONFIG["cv_folds"]):
        key = (sid, fold_id)
        split = LOCKED_SPLITS[key]
        train_indices, test_indices = split["train_indices"], split["test_indices"]
        shallow = SHALLOW_BY_FOLD[key]
        if not np.array_equal(shallow["test_indices"], test_indices) or not np.array_equal(shallow["labels"], labels[test_indices]):
            raise AssertionError(f"Raw/artifact label or test-index mismatch for {sid} fold {fold_id}")
        by_scale, equal_probabilities, equal_scores, margin_scales = fit_short_scale_riemann(covariances, labels, train_indices, test_indices)
        probabilities_by_method = {
            "shallow_locked": shallow["probabilities"],
            "riemann_1s": by_scale["1s"]["probabilities"],
            "riemann_2s": by_scale["2s"]["probabilities"],
            "riemann_equal_short_scales": equal_probabilities,
            "shallow_riemann_fusion": fixed_fusion(shallow["probabilities"], equal_probabilities),
        }
        for view_id, fitted_scale in margin_scales.items():
            margin_rows.append({"subject_id": sid, "fold_id": fold_id, "view_id": view_id, "training_margin_scale": float(fitted_scale), "fit_scope": "outer_train_only"})
        for method, probabilities in probabilities_by_method.items():
            if probabilities.shape != (len(test_indices), 2) or not np.all(np.isfinite(probabilities)):
                raise ValueError(f"Invalid {method} probabilities for {sid} fold {fold_id}")
            seen[method][test_indices] += 1
            result = fold_metric_record(sid, fold_id, method, train_indices, test_indices, labels[test_indices], probabilities)
            fold_results.append(result)
            for row_index, trial_index in enumerate(test_indices):
                prediction_rows.append({"subject_id": sid, "fold_id": fold_id, "trial_index": int(trial_index), "method": method, "true_label": int(labels[trial_index]), "prediction": int(result["predictions"][row_index]), "probability_0": float(probabilities[row_index, 0]), "probability_1": float(probabilities[row_index, 1]), "outer_train_only": True})
    for method, counts in seen.items():
        if not np.all(counts == 1):
            raise AssertionError(f"{sid} {method} does not have exactly-once OOF coverage: {counts.tolist()}")
    return fold_results, prediction_rows, margin_rows, covariance_diagnostics

## 5.2 Run All Subjects

In [ ]:
print("=" * 72)
print(json.dumps(CONFIG, indent=2, sort_keys=True))
print("=" * 72)

FOLD_RESULTS, PREDICTION_ROWS, MARGIN_ROWS, COVARIANCE_DIAGNOSTICS = [], [], [], []
SUBJECTS, inventory = [], []
for path in selected_files():
    subject_data = load_subject_raw(path)
    sid = subject_data["subject_id"]
    fold_results, prediction_rows, margin_rows, covariance_diagnostics = run_subject(subject_data)
    SUBJECTS.append(sid)
    FOLD_RESULTS.extend(fold_results)
    PREDICTION_ROWS.extend(prediction_rows)
    MARGIN_ROWS.extend(margin_rows)
    for row in covariance_diagnostics:
        COVARIANCE_DIAGNOSTICS.append({"subject_id": sid, **row})
    inventory.append({"subject_id": sid, "n_trials": len(subject_data["labels"]), "class_0": int(np.sum(subject_data["labels"] == 0)), "class_1": int(np.sum(subject_data["labels"] == 1)), "marker_fallback_count": subject_data["fallback_count"], "source_path": subject_data["source_path"], "source_sha256": subject_data["source_sha256"], "subject_split_hash": CONFIG["expected_global_split_hash"]})
    print(f"{sid}: completed fixed same-fold fusion")

if SUBJECTS != EXPECTED_SUBJECTS or len(FOLD_RESULTS) != 50 * 5 * len(METHODS) or len(PREDICTION_ROWS) != 50 * 40 * len(METHODS):
    raise AssertionError("Full50 output cardinality mismatch")
subject_inventory_path = ARTIFACT_DIR / "subject_inventory.csv"
pd.DataFrame(inventory).to_csv(subject_inventory_path, index=False)

# 6. Results
## 6.1 Pooled Subject OOF and Global Metrics

The primary estimand is each subject's pooled exactly-once OOF balanced accuracy, averaged across subjects. Statistical comparisons pair subjects, never folds.

In [ ]:
def bootstrap_mean_ci(values, seed, iterations):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(values, size=(iterations, len(values)), replace=True).mean(axis=1)
    return [float(x) for x in np.percentile(draws, [2.5, 97.5])]

PREDICTIONS = pd.DataFrame(PREDICTION_ROWS).sort_values(["subject_id", "method", "trial_index"]).reset_index(drop=True)
subject_rows = []
for (sid, method), rows in PREDICTIONS.groupby(["subject_id", "method"], sort=True):
    rows = rows.sort_values("trial_index")
    if rows["trial_index"].tolist() != list(range(40)) or rows["trial_index"].nunique() != 40:
        raise AssertionError(f"Incomplete pooled OOF rows for {sid} {method}")
    subject_rows.append({"subject_id": sid, "method": method, "n_trials": len(rows), "accuracy": float(accuracy_score(rows["true_label"], rows["prediction"])), "balanced_accuracy": float(balanced_accuracy_score(rows["true_label"], rows["prediction"])), "confusion_matrix": confusion_matrix(rows["true_label"], rows["prediction"], labels=[0, 1]).tolist()})
SUBJECT_METRICS = pd.DataFrame(subject_rows)

GLOBAL_METRICS = {"primary_estimand": "mean_subject_pooled_exactly_once_oof_balanced_accuracy", "n_subjects": 50, "n_trials": 2000, "n_folds": 250, "global_split_hash": CONFIG["expected_global_split_hash"], "methods": {}}
for method in METHODS:
    values = SUBJECT_METRICS.loc[SUBJECT_METRICS.method == method, "balanced_accuracy"].to_numpy()
    GLOBAL_METRICS["methods"][method] = {"mean_subject_balanced_accuracy": float(values.mean()), "subject_bootstrap_95_ci": bootstrap_mean_ci(values, CONFIG["bootstrap_seed"], CONFIG["bootstrap_iterations"]), "n_subjects": len(values)}

def paired_comparison(method, reference="shallow_locked"):
    table = SUBJECT_METRICS.pivot(index="subject_id", columns="method", values="balanced_accuracy").loc[EXPECTED_SUBJECTS]
    delta = (table[method] - table[reference]).to_numpy()
    statistic, p_value = (0.0, 1.0) if np.allclose(delta, 0.0) else wilcoxon(delta, alternative="two-sided", zero_method="wilcox")
    rng = np.random.default_rng(CONFIG["bootstrap_seed"])
    draws = rng.choice(delta, size=(CONFIG["bootstrap_iterations"], len(delta)), replace=True).mean(axis=1)
    return {"method": method, "reference": reference, "mean_delta": float(delta.mean()), "mean_delta_points": float(100.0 * delta.mean()), "paired_subject_bootstrap_95_ci": [float(x) for x in np.percentile(draws, [2.5, 97.5])], "paired_subject_bootstrap_95_ci_points": [float(100.0 * x) for x in np.percentile(draws, [2.5, 97.5])], "wilcoxon_statistic": float(statistic), "wilcoxon_p_value": float(p_value), "wins": int(np.sum(delta > 0)), "ties": int(np.sum(delta == 0)), "losses": int(np.sum(delta < 0)), "n_subjects": len(delta), "inference_unit": "subject"}

PAIRED_STATS = {method: paired_comparison(method) for method in METHODS if method != "shallow_locked"}
GLOBAL_METRICS["paired_vs_shallow"] = PAIRED_STATS
GLOBAL_METRICS["protocol"] = {"shallow_retrained": False, "shallow_selected": False, "riemann_outer_train_only": True, "outer_test_used_for_selection": False, "repeated_60_40_consumed": False, "learned_stack": False, "fusion_weights": {"shallow": CONFIG["shallow_weight"], "riemann_equal_short_scales": CONFIG["riemann_weight"]}, "riemann_probability_mapping": CONFIG["riemann_probability_mapping"]}
print(json.dumps(GLOBAL_METRICS, indent=2))

## 6.2 Performance Visualizations

In [ ]:
plot_labels = {"shallow_locked": "Locked Shallow", "riemann_1s": "Riemann 1 s", "riemann_2s": "Riemann 2 s", "riemann_equal_short_scales": "Equal Riemann", "shallow_riemann_fusion": "50/50 Fusion"}

fig, ax = plt.subplots(figsize=(14, 6))
pivot = SUBJECT_METRICS.pivot(index="subject_id", columns="method", values="balanced_accuracy").loc[EXPECTED_SUBJECTS]
x = np.arange(len(pivot))
for method, marker in [("shallow_locked", "o"), ("riemann_equal_short_scales", "s"), ("shallow_riemann_fusion", "^")]:
    ax.plot(x, 100 * pivot[method], marker=marker, markersize=3, linewidth=0.8, label=plot_labels[method])
ax.axhline(50, color="black", linestyle="--", linewidth=1, label="Chance")
ax.set(xlabel="Subject", ylabel="Pooled OOF balanced accuracy (%)", title="Liu2024 Same-Fold Locked Shallow and Riemann Fusion")
ax.set_xticks(x[::2], [sid.replace("sub-", "") for sid in EXPECTED_SUBJECTS][::2])
ax.legend(ncol=4)
fig.tight_layout()
subject_performance_plot_path = ARTIFACT_DIR / "same_fold_fusion_subject_performance.png"
fig.savefig(subject_performance_plot_path, dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
means = [100 * GLOBAL_METRICS["methods"][method]["mean_subject_balanced_accuracy"] for method in METHODS]
cis = [GLOBAL_METRICS["methods"][method]["subject_bootstrap_95_ci"] for method in METHODS]
errors = np.asarray([[mean - 100 * ci[0], 100 * ci[1] - mean] for mean, ci in zip(means, cis)]).T
bars = ax.bar(np.arange(len(METHODS)), means, yerr=errors, capsize=4, color=["#355070", "#6d597a", "#b56576", "#e56b6f", "#eaac8b"])
ax.axhline(50, color="black", linestyle="--", linewidth=1)
ax.set(ylabel="Mean subject balanced accuracy (%)", title="Same-Fold Method Comparison")
ax.set_xticks(np.arange(len(METHODS)), [plot_labels[m] for m in METHODS], rotation=20, ha="right")
for bar, value in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.5, f"{value:.2f}", ha="center", fontsize=9)
fig.tight_layout()
global_performance_plot_path = ARTIFACT_DIR / "same_fold_fusion_global_performance.png"
fig.savefig(global_performance_plot_path, dpi=180)
plt.close(fig)

fused_rows = PREDICTIONS[PREDICTIONS.method == "shallow_riemann_fusion"]
matrix = confusion_matrix(fused_rows.true_label, fused_rows.prediction, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(matrix[i, j]), ha="center", va="center")
ax.set(xticks=[0, 1], yticks=[0, 1], xlabel="Predicted", ylabel="True", title="50/50 Fusion Aggregated OOF Confusion Matrix")
fig.colorbar(image, ax=ax)
fig.tight_layout()
confusion_plot_path = ARTIFACT_DIR / "same_fold_fusion_aggregated_confusion_matrix.png"
fig.savefig(confusion_plot_path, dpi=180)
plt.close(fig)

## 6.3 Experiment Summary

In [ ]:
print("Primary same-fold fusion summary")
for method in METHODS:
    metrics = GLOBAL_METRICS["methods"][method]
    print(f"{plot_labels[method]}: {100 * metrics['mean_subject_balanced_accuracy']:.2f}% BA, 95% CI {100 * metrics['subject_bootstrap_95_ci'][0]:.2f}-{100 * metrics['subject_bootstrap_95_ci'][1]:.2f}%")
print(f"Fusion vs Shallow: {PAIRED_STATS['shallow_riemann_fusion']['mean_delta_points']:+.2f} points, p={PAIRED_STATS['shallow_riemann_fusion']['wilcoxon_p_value']:.6g}")

## 6.4 Leakage Assertions

In [ ]:
LEAKAGE_ASSERTIONS = {
    "locked_shallow_artifact_validated_before_raw_processing": True,
    "all_50_subjects_present": SUBJECTS == EXPECTED_SUBJECTS,
    "forty_unique_test_trials_per_subject_method": bool(PREDICTIONS.groupby(["subject_id", "method"])["trial_index"].agg(["count", "nunique"]).eq(40).all().all()),
    "liu29_provenance_exact": SHALLOW_PROVENANCE["channel_names"] == LIU29,
    "split_hash_exact": SHALLOW_PROVENANCE["global_split_hash"] == CONFIG["expected_global_split_hash"],
    "all_outer_train_test_disjoint": all(row["no_overlap"] for row in SPLIT_ASSERTIONS),
    "shallow_retrained": False,
    "shallow_selected": False,
    "outer_test_used_for_selection": False,
    "outer_test_used_for_transform_fit": False,
    "repeated_60_40_predictions_or_transforms_consumed": False,
    "learned_stack_present": False,
    "trial_filters_cross_boundaries": False,
    "covariance_scope": "single_trial_only",
    "tangent_scaler_classifier_margin_scale_fit_scope": "outer_train_only",
}
required_true = ["locked_shallow_artifact_validated_before_raw_processing", "all_50_subjects_present", "forty_unique_test_trials_per_subject_method", "liu29_provenance_exact", "split_hash_exact", "all_outer_train_test_disjoint"]
required_false = ["shallow_retrained", "shallow_selected", "outer_test_used_for_selection", "outer_test_used_for_transform_fit", "repeated_60_40_predictions_or_transforms_consumed", "learned_stack_present", "trial_filters_cross_boundaries"]
if not all(LEAKAGE_ASSERTIONS[key] is True for key in required_true) or not all(LEAKAGE_ASSERTIONS[key] is False for key in required_false):
    raise AssertionError(f"Leakage assertion failed: {LEAKAGE_ASSERTIONS}")
print(json.dumps(LEAKAGE_ASSERTIONS, indent=2))

## 6.5 Save Artifacts

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, "w") as f:
    json.dump(FOLD_RESULTS, f, indent=2, allow_nan=False)

fold_metrics_path = ARTIFACT_DIR / "fold_metrics.csv"
pd.DataFrame([{k: v for k, v in row.items() if k not in {"train_indices", "test_indices", "true_labels", "predictions", "probabilities", "confusion_matrix", "prediction_histogram", "collapse_diagnostics"}} for row in FOLD_RESULTS]).to_csv(fold_metrics_path, index=False)

subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, "w") as f:
    json.dump(subject_rows, f, indent=2, allow_nan=False)
subject_metrics_csv_path = ARTIFACT_DIR / "subject_metrics.csv"
SUBJECT_METRICS.assign(confusion_matrix=SUBJECT_METRICS.confusion_matrix.map(json.dumps)).to_csv(subject_metrics_csv_path, index=False)

global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, "w") as f:
    json.dump(GLOBAL_METRICS, f, indent=2, allow_nan=False)

predictions_path = ARTIFACT_DIR / "predictions.csv"
PREDICTIONS.to_csv(predictions_path, index=False)
paired_stats_path = ARTIFACT_DIR / "paired_subject_statistics.json"
with open(paired_stats_path, "w") as f:
    json.dump(PAIRED_STATS, f, indent=2, allow_nan=False)
split_assertions_path = ARTIFACT_DIR / "split_assertions.json"
with open(split_assertions_path, "w") as f:
    json.dump({"global_split_hash": CONFIG["expected_global_split_hash"], "fold_assertions": SPLIT_ASSERTIONS, "leakage_assertions": LEAKAGE_ASSERTIONS}, f, indent=2, allow_nan=False)
split_indices_path = ARTIFACT_DIR / "split_indices.json"
with open(split_indices_path, "w") as f:
    json.dump({sid: [{"fold_id": fold_id, "train_indices": LOCKED_SPLITS[(sid, fold_id)]["train_indices"].tolist(), "test_indices": LOCKED_SPLITS[(sid, fold_id)]["test_indices"].tolist()} for fold_id in range(5)] for sid in EXPECTED_SUBJECTS}, f, indent=2)
shallow_provenance_path = ARTIFACT_DIR / "locked_shallow_provenance.json"
with open(shallow_provenance_path, "w") as f:
    json.dump(SHALLOW_PROVENANCE, f, indent=2)
margin_scales_path = ARTIFACT_DIR / "training_margin_scales.csv"
pd.DataFrame(MARGIN_ROWS).to_csv(margin_scales_path, index=False)
covariance_diagnostics_path = ARTIFACT_DIR / "covariance_diagnostics.csv"
pd.DataFrame(COVARIANCE_DIAGNOSTICS).to_csv(covariance_diagnostics_path, index=False)

artifact_paths = {
    "config": str(config_path), "run_log": str(LOG_PATH), "cv_results": str(cv_results_path), "fold_metrics": str(fold_metrics_path), "subject_metrics_json": str(subject_metrics_path), "subject_metrics_csv": str(subject_metrics_csv_path), "global_metrics": str(global_metrics_path), "predictions": str(predictions_path), "paired_subject_statistics": str(paired_stats_path), "split_assertions": str(split_assertions_path), "split_indices": str(split_indices_path), "locked_shallow_provenance": str(shallow_provenance_path), "subject_inventory": str(subject_inventory_path), "training_margin_scales": str(margin_scales_path), "covariance_diagnostics": str(covariance_diagnostics_path), "subject_performance_plot": str(subject_performance_plot_path), "global_performance_plot": str(global_performance_plot_path), "aggregated_confusion_plot": str(confusion_plot_path)
}
run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "experiment_name": CONFIG["experiment_name"],
    "config_note": CONFIG["config_note"],
    "subjects": SUBJECTS,
    "n_channels": len(MOTOR13),
    "channel_names": MOTOR13,
    "source_montage_name": "liu29",
    "source_channel_names": LIU29,
    "spatial_mode": CONFIG["spatial_mode"],
    "filter_context": CONFIG["filter_context"],
    "covariance_estimator": CONFIG["covariance_estimator"],
    "base_classifier": CONFIG["base_classifier"],
    "view_specs": VIEW_SPECS,
    "probability_and_fusion": {"view_score": "outer-training-margin-normalized LDA decision score", "within_scale": "equal mean score over fixed windows and bands", "riemann_short_scale": "equal mean of 1s and 2s scores", "probability_mapping": CONFIG["riemann_probability_mapping"], "primary_fusion": "0.5 locked Shallow + 0.5 equal short-scale Riemann probabilities"},
    "locked_shallow_provenance": SHALLOW_PROVENANCE,
    "leakage_assertions": LEAKAGE_ASSERTIONS,
    "seeds": {"base": BASE_SEED, "bootstrap": CONFIG["bootstrap_seed"]},
    "global_metrics": GLOBAL_METRICS,
    "artifacts": artifact_paths,
}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2, allow_nan=False)

print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")

try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass